# 라이브러리 import

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 모델 불러오기 (Qwen2.5-0.5B-Instruct)

## Qwen2.5-0.5B-Instruct 요약

### 📦 모델 크기

* 약 **0.5B (4.9억 파라미터)**
* Qwen2.5 계열 중 가장 작은 초경량 모델

### 💻 로컬 실행

* **CPU / 저사양 GPU에서도 실행 가능**
* 4~8GB VRAM 환경에서도 4bit 양자화로 구동 가능
* 가벼워서 로컬 LLM 입문용, 테스트용으로 적합

### 🇰🇷 한국어 성능

* 기본적인 한국어 대화/이해 가능
* 짧은 요약, 간단한 질문 응답은 무난
* 다만:

  * 자연스러움 부족
  * 복잡한 문장 생성 및 추론은 약함

### 📜 라이선스

* **Apache License 2.0 (오픈소스 라이선스)** ([Hugging Face][1])
* 의미:

  * 무료 다운로드 및 사용 가능
  * 상업적 사용 가능
  * 수정 및 재배포 가능
  * 단, 저작권 고지 및 라이선스 사본 유지 필요

---

### 🧾 정리

> **아주 가볍고 자유롭게 쓸 수 있는 초소형 LLM이지만, 한국어/추론 성능은 제한적**

[1]: https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/c89bee90d9f811437d9735454613c35b4a3c4dc8/LICENSE?utm_source=chatgpt.com "LICENSE · Qwen/Qwen2.5-0.5B-Instruct at c89bee90d9f811437d9735454613c35b4a3c4dc8"


In [2]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

# 생성 테스트

In [13]:
prompt = "연필"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

연필을 사용해보세요. 이전에 작성한 모든 글을 복사해서 여기에


In [14]:
prompt = "연필"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :]
probs = torch.softmax(logits, dim=-1)

top_probs, top_ids = torch.topk(probs, 10)

for prob, token_id in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode([token_id])
    print(f"{repr(token):15} {prob.item()*100:.2f}%")

'을'             6.08%
','             4.17%
'에서'            3.69%
'의'             3.47%
':'             2.23%
' -'            2.23%
'과'             2.23%
'('             1.86%
' |'            1.75%
'은'             1.75%


# 전시용 코드 예시

토큰 입력
   ->
모델 추론
   ->
다음 토큰 확률 계산
   ->
Top-k 선택
   ->
다음 토큰 결정
   ->
문장에 추가
   ->
새 입력
   ->
반복

In [15]:
prompt = input("Input: ")

max_steps = 5  # 전시에서는 5~15 추천
top_k = 5

for step in range(max_steps):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]          # 마지막 토큰 기준
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_ids = torch.topk(probs, k=top_k)

    print("\n" + "="*50)
    print(f"[{step+1}단계] 현재 문장:")
    print(prompt)
    print("\nTop-k next tokens:")

    candidates = []

    for i in range(top_k):
        token_id = top_ids[0][i].item()
        prob = top_probs[0][i].item()
        token = tokenizer.decode([token_id])

        candidates.append((token, prob, token_id))

        print(f"{i+1:2d}. {repr(token):10s}  {prob*100:6.2f}%")

    # 가장 확률 높은 토큰 선택 (greedy)
    next_token = candidates[0][0]
    
    # 문장 이어붙이기
    prompt += next_token

    print("\nSelected:", repr(next_token))

Input:  연필



[1단계] 현재 문장:
연필

Top-k next tokens:
 1. '을'           6.08%
 2. ','           4.17%
 3. '에서'          3.69%
 4. '의'           3.47%
 5. ':'           2.23%

Selected: '을'

[2단계] 현재 문장:
연필을

Top-k next tokens:
 1. ' 사용'        19.24%
 2. ' �'          8.01%
 3. ' 이용'         5.86%
 4. ' 만'          1.78%
 5. ' �'          1.67%

Selected: ' 사용'

[3단계] 현재 문장:
연필을 사용

Top-k next tokens:
 1. '하는'         23.24%
 2. '하여'         16.99%
 3. '한'          12.40%
 4. '해'           7.08%
 5. '하고'          4.30%

Selected: '하는'

[4단계] 현재 문장:
연필을 사용하는

Top-k next tokens:
 1. ' 것은'         7.67%
 2. ' �'          4.37%
 3. ' �'          4.10%
 4. ' 것이'         4.10%
 5. ' 사람'         3.86%

Selected: ' 것은'

[5단계] 현재 문장:
연필을 사용하는 것은

Top-k next tokens:
 1. ' �'          2.06%
 2. ' �'          1.94%
 3. ' 어'          1.94%
 4. ' 매우'         1.94%
 5. ' �'          1.82%

Selected: ' �'
